In [20]:
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

In [12]:
df = pd.read_csv('data.csv')
df = df.sort_values(by=['Year', 'Share'], ascending=[True, False])
df

,Player,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,...,OWS,DWS,WS,WS/48,OBPM,DBPM,BPM,VORP,Share,Year
0,Kevin Garnett,82.0,82.0,3231.0,804.0,1611.0,0.499,11.0,43.0,0.256,...,10.4,8.0,18.3,0.272,6.8,3.3,10.2,10.0,0.991,2004
14,Tim Duncan,69.0,68.0,2527.0,592.0,1181.0,0.501,2.0,12.0,0.167,...,5.9,7.2,13.1,0.249,5.2,3.3,8.5,6.7,0.582,2004
12,Jermaine O'Neal,78.0,78.0,2788.0,608.0,1400.0,0.434,2.0,18.0,0.111,...,2.7,6.3,9.0,0.155,1.1,1.3,2.4,3.1,0.425,2004
1,Peja Stojaković,81.0,81.0,3264.0,665.0,1386.0,0.480,240.0,554.0,0.433,...,11.4,2.1,13.5,0.198,5.2,-1.3,3.9,4.9,0.228,2004
13,Kobe Bryant,65.0,64.0,2447.0,516.0,1178.0,0.438,71.0,217.0,0.327,...,7.8,3.0,10.7,0.210,5.1,0.5,5.6,4.7,0.172,2004
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10343,Danny Green,2.0,0.0,18.0,0.0,2.0,0.000,0.0,1.0,0.000,...,0.0,0.0,0.0,-0.035,-7.4,0.8,-6.6,0.0,0.000,2024
10344,Ron Harper Jr.,1.0,0.0,4.0,0.0,0.0,0.000,0.0,0.0,0.000,...,0.0,0.0,0.0,0.087,-11.7,2.8,-8.8,0.0,0.000,2024
10345,Justin Jackson,2.0,0.0,1.0,0.0,0.0,0.000,0.0,0.0,0.000,...,0.0,0.0,0.0,0.031,-6.3,-1.2,-7.5,0.0,0.000,2024
10346,Dmytro Skapintsev,2.0,0.0,2.0,0.0,1.0,0.000,0.0,0.0,0.000,...,0.0,0.0,0.0,-0.483,-16.0,-9.8,-25.9,0.0,0.000,2024


In [21]:
# Experiment Cell
pd.concat([df[df['Year'] == 2004], df[df['Year'] == 2005]])
features = df.columns.to_list()
features = [feature for feature in df.columns.to_list() if feature not in ['Player', 'Share', 'Year']]
df[features]
df[df['Year'] == 2024]

,Player,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,...,OWS,DWS,WS,WS/48,OBPM,DBPM,BPM,VORP,Share,Year
9780,Nikola Jokić,79.0,79.0,2737.0,822.0,1411.0,0.583,83.0,231.0,0.359,...,12.0,5.1,17.0,0.299,9.0,4.2,13.2,10.6,0.935,2024
9777,Shai Gilgeous-Alexander,75.0,75.0,2553.0,796.0,1487.0,0.535,95.0,269.0,0.353,...,10.5,4.2,14.6,0.275,6.7,2.3,9.0,7.1,0.646,2024
9776,Luka Dončić,70.0,70.0,2624.0,804.0,1652.0,0.487,284.0,744.0,0.382,...,8.5,3.5,12.0,0.220,8.3,1.7,9.9,8.0,0.572,2024
9778,Giannis Antetokounmpo,73.0,73.0,2567.0,837.0,1369.0,0.611,34.0,124.0,0.274,...,9.5,3.7,13.2,0.246,6.7,2.4,9.0,7.2,0.194,2024
9779,Jalen Brunson,77.0,77.0,2726.0,790.0,1648.0,0.479,211.0,526.0,0.401,...,8.8,2.4,11.2,0.198,6.3,-0.4,5.8,5.4,0.143,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10343,Danny Green,2.0,0.0,18.0,0.0,2.0,0.000,0.0,1.0,0.000,...,0.0,0.0,0.0,-0.035,-7.4,0.8,-6.6,0.0,0.000,2024
10344,Ron Harper Jr.,1.0,0.0,4.0,0.0,0.0,0.000,0.0,0.0,0.000,...,0.0,0.0,0.0,0.087,-11.7,2.8,-8.8,0.0,0.000,2024
10345,Justin Jackson,2.0,0.0,1.0,0.0,0.0,0.000,0.0,0.0,0.000,...,0.0,0.0,0.0,0.031,-6.3,-1.2,-7.5,0.0,0.000,2024
10346,Dmytro Skapintsev,2.0,0.0,2.0,0.0,1.0,0.000,0.0,0.0,0.000,...,0.0,0.0,0.0,-0.483,-16.0,-9.8,-25.9,0.0,0.000,2024


In [14]:
def df_concat_stats(df, startYear, endYear):
    final_df = pd.DataFrame()
    if startYear == endYear:
        return df
    while startYear <= endYear:
        if final_df.empty:
            final_df = df[df['Year'] == startYear]
        else:
            final_df = pd.concat([final_df, df[df['Year'] == startYear]])
        startYear +=1
    return final_df

# TESTS #
test = df_concat_stats(df, 2004, 2004)
test.iloc[0:5]['Player'].tolist()

['Kevin Garnett',
 'Tim Duncan',
 "Jermaine O'Neal",
 'Peja Stojaković',
 'Kobe Bryant']

In [19]:
years = range(2005, 2025)

results = []

features = [feature for feature in df.columns.to_list() if feature not in ['Player', 'Share', 'Year']]

right = 0
wrong = 0

for year in years:
    model = XGBRegressor(random_state=42)

    train_df = df_concat_stats(df, 2004, year-1)
    test_df = df[df['Year'] == year]
    
    X_train = train_df[features]
    y_train = train_df['Share']
    
    X_test = test_df[features]
    y_test = test_df['Share']

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_pred, y_test)
    
    test_copy = test_df.copy()
    test_copy['Predicted Share'] = y_pred
    actual_mvp = test_copy.sort_values('Share', ascending=False).iloc[0]['Player']
    predicted_mvp = test_copy.sort_values('Predicted Share', ascending=False).iloc[0]['Player']
    top_five = test_copy.sort_values('Predicted Share', ascending=False).iloc[0:5]['Player'].to_list()
    
    
    results.append({
        "Train Years": [2004, year-1],
        "Test Year": year,
        "MSE": mse,
        "Predicted MVP": predicted_mvp,
        "Actual MVP": actual_mvp,
        "Top Five": top_five
    })

    
    if predicted_mvp == actual_mvp:
        right += 1
    else:
        wrong += 1

print("Right: " + str(right) + ", Wrong: " + str(wrong))
results







Right: 10, Wrong: 10


[{'Train Years': [2004, 2004],
  'Test Year': 2005,
  'MSE': 7.256673105584919e-08,
  'Predicted MVP': 'Steve Nash',
  'Actual MVP': 'Steve Nash',
  'Top Five': ['Steve Nash',
   "Shaquille O'Neal",
   'Dirk Nowitzki',
   'Tim Duncan',
   'Allen Iverson']},
 {'Train Years': [2004, 2005],
  'Test Year': 2006,
  'MSE': 0.001999417547542164,
  'Predicted MVP': 'Ray Allen',
  'Actual MVP': 'Steve Nash',
  'Top Five': ['Ray Allen',
   'Elton Brand',
   'Steve Nash',
   'Kevin Garnett',
   'Dirk Nowitzki']},
 {'Train Years': [2004, 2006],
  'Test Year': 2007,
  'MSE': 0.00103917250875013,
  'Predicted MVP': 'Steve Nash',
  'Actual MVP': 'Dirk Nowitzki',
  'Top Five': ['Steve Nash',
   'Dirk Nowitzki',
   'Tim Duncan',
   'Dwyane Wade',
   'Kobe Bryant']},
 {'Train Years': [2004, 2007],
  'Test Year': 2008,
  'MSE': 0.0028254075478178087,
  'Predicted MVP': 'Chris Paul',
  'Actual MVP': 'Kobe Bryant',
  'Top Five': ['Chris Paul',
   'Chauncey Billups',
   'LeBron James',
   "Amar'e Stoudemire